# 11 — Synthetic French Phishing Generation

**C1 Source Type:** `Base de données` (preparation step)

---

## Objective

Generate **purely synthetic French phishing and legitimate emails** using
template engines + `Faker(fr_FR)`. This supplements the culturally-adapted
emails from notebook 10 and together they form the **raw data** that feeds
the SQLite database in notebook 09.

### Why synthetic data?

| Reason | Detail |
|--------|--------|
| **No public French phishing corpus exists** | We must create our own |
| **Control over class balance** | We can generate exact phishing-to-legit ratios |
| **Entity coverage** | Ensure all 8 french archetypes are represented |
| **Augmentation for fine-tuning** | More French data → better CamemBERTv2 |
| **RGPD-safe** | Synthetic data contains no real PII |

### Input → Output

- **Input:** Template definitions + Faker(fr_FR)
- **Output:** `data/raw/db/synthetic_fr_phishing.csv` (2 000+ phishing + 1 000+ legit)

> This output feeds **notebook 09** (DB extraction) as raw data for the SQLite database.

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
import random
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from faker import Faker

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_DIR: Path = Path("data/raw/db")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED: int = 73  # Different seed from notebook 10 for diversity
random.seed(SEED)
fake = Faker("fr_FR")
Faker.seed(SEED)

# Target volumes
PHISHING_TARGET: int = 2_000
LEGIT_TARGET: int = 1_000

print(f"Output dir       : {OUTPUT_DIR.resolve()}")
print(f"Phishing target  : {PHISHING_TARGET:,}")
print(f"Legit target     : {LEGIT_TARGET:,}")
print(f"Seed             : {SEED}")

Output dir       : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/db
Phishing target  : 2,000
Legit target     : 1,000
Seed             : 73


## 1. Utility Functions

Shared helpers for generating realistic French administrative data.

In [2]:
# ── Shared helpers ────────────────────────────────────────────────────

def _ref() -> str:
    """Generate a fake French admin reference number."""
    prefix: str = random.choice(["REF", "DOS", "N°", "DEM", "FAC", "CMD"])
    return f"{prefix}-{fake.numerify('####-####-##')}"

def _amount(low: float = 12.50, high: float = 3500.00) -> str:
    """Generate a realistic French currency amount."""
    val: float = round(random.uniform(low, high), 2)
    return f"{val:,.2f} €".replace(",", " ").replace(".", ",")

def _small_amount() -> str:
    """Small fee (delivery, customs, etc.)."""
    return _amount(0.99, 4.99)

def _date_fr() -> str:
    """Generate a date in French format."""
    return fake.date_between(start_date="-30d", end_date="+15d").strftime("%d/%m/%Y")

def _past_date() -> str:
    """Generate a past date."""
    return fake.date_between(start_date="-90d", end_date="-1d").strftime("%d/%m/%Y")

def _deadline() -> str:
    """Generate an urgency deadline."""
    return f"{random.choice([24, 48, 72])} heures"

def _siret() -> str:
    """Generate a fake SIRET number (14 digits)."""
    return fake.numerify('### ### ### #####')

def _secu() -> str:
    """Generate a fake French social security number."""
    return fake.numerify('# ## ## ## ### ### ##')

def _iban_fr() -> str:
    """Generate a fake French IBAN."""
    return f"FR{fake.numerify('## #### #### #### #### #### ###')}"

def _bank() -> str:
    """Pick a random French bank."""
    return random.choice([
        "BNP Paribas", "Crédit Agricole", "Société Générale",
        "LCL", "Banque Populaire", "Caisse d'Épargne", "CIC",
        "La Banque Postale", "Boursorama", "Crédit Mutuel",
    ])

def _telecom() -> str:
    """Pick a random French telecom operator."""
    return random.choice(["SFR", "Orange", "Free", "Bouygues Telecom"])

def _city() -> str:
    """French city name."""
    return fake.city()

print("Utility functions loaded.")
print(f"Sample ref    : {_ref()}")
print(f"Sample amount : {_amount()}")
print(f"Sample SIRET  : {_siret()}")
print(f"Sample sécu   : {_secu()}")
print(f"Sample IBAN   : {_iban_fr()}")

Utility functions loaded.
Sample ref    : N°-4187-9274-91
Sample amount : 438,71 €
Sample SIRET  : 768 837 104 60658
Sample sécu   : 6 78 62 04 993 175 42
Sample IBAN   : FR30 3440 3829 6111 3282 7778 354


## 2. Phishing Templates — French Archetypes

Each archetype has **3-5 templates** for maximum diversity.
Templates use varied structures: formal letters, urgent alerts, fake confirmations.

In [3]:
# ── PHISHING TEMPLATES ────────────────────────────────────────────────
#
# Each template returns (subject, body, archetype, fr_entity)

PHISHING_TEMPLATES: list = [

    # ── DGFiP / Impôts ────────────────────────────────────────────────
    lambda: (
        f"Rappel fiscal — Montant restant dû {_ref()}",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Votre avis d'imposition pour l'année 2025 fait apparaître un solde "
        f"débiteur de {_amount()}. Ce montant doit être réglé avant le {_date_fr()}.\n\n"
        f"En cas de non-paiement, des majorations de retard de 10 % seront "
        f"appliquées automatiquement (article 1727 du CGI).\n\n"
        f"Accédez à votre espace pour régulariser :\n"
        f"https://impots-gouv-fr-paiement.com/{fake.lexify('????????')}\n\n"
        f"Direction Générale des Finances Publiques",
        "dgfip_tax", "DGFiP"
    ),
    lambda: (
        f"Trop-perçu fiscal — Remboursement de {_amount(150, 680)}",
        f"Cher(e) contribuable,\n\n"
        f"Après vérification de votre dossier fiscal ({_ref()}), un trop-perçu "
        f"de {_amount(150, 680)} a été identifié.\n\n"
        f"Pour recevoir ce remboursement sous 5 jours ouvrés, confirmez vos "
        f"coordonnées bancaires (RIB/IBAN) :\n"
        f"https://dgfip-remboursement-fr.com/{fake.lexify('??????')}\n\n"
        f"Offre valable {_deadline()}.\n\n"
        f"DGFiP — Service des Remboursements",
        "dgfip_tax", "DGFiP"
    ),
    lambda: (
        f"Convocation pour vérification fiscale",
        f"Madame, Monsieur,\n\n"
        f"Votre entreprise (SIRET : {_siret()}) a été sélectionnée pour un "
        f"contrôle fiscal au titre des exercices 2023-2025.\n\n"
        f"Veuillez préparer et transmettre les documents suivants sous {_deadline()} :\n"
        f"- Livre des recettes et des dépenses\n"
        f"- Factures de vente et d'achat\n"
        f"- Relevés bancaires professionnels\n\n"
        f"Déposez vos documents ici :\n"
        f"https://verification-fiscale-gouv.fr/{fake.lexify('????????')}\n\n"
        f"Service de Vérification Nationale\nDGFiP",
        "dgfip_tax", "DGFiP"
    ),

    # ── URSSAF ────────────────────────────────────────────────────────
    lambda: (
        f"Cotisations impayées — Mise en demeure {_ref()}",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Nous constatons que vos cotisations sociales du trimestre en cours "
        f"(montant : {_amount()}) n'ont pas été réglées à la date d'échéance.\n\n"
        f"Numéro de cotisant : {fake.numerify('### ### ### ###')}\n"
        f"SIRET : {_siret()}\n\n"
        f"Sans régularisation dans les {_deadline()}, votre dossier sera "
        f"transmis au service de recouvrement forcé et des majorations "
        f"de 5 % par mois seront appliquées.\n\n"
        f"Régularisez en ligne :\n"
        f"https://urssaf-cotisations.fr/{fake.lexify('????????')}\n\n"
        f"URSSAF — Service Recouvrement\n{_city()}",
        "urssaf_cotisation", "URSSAF"
    ),
    lambda: (
        f"Déclaration trimestrielle — Rappel urgent",
        f"Bonjour,\n\n"
        f"Votre déclaration trimestrielle de chiffre d'affaires est en retard. "
        f"La date limite était le {_past_date()}.\n\n"
        f"En l'absence de déclaration sous {_deadline()}, nous serons contraints "
        f"de procéder à une taxation d'office majorée de 15 %.\n\n"
        f"→ Déclarer maintenant : https://autoentrepreneur-urssaf.fr/{fake.lexify('??????')}\n\n"
        f"URSSAF Île-de-France",
        "urssaf_cotisation", "URSSAF"
    ),
    lambda: (
        f"Attestation de vigilance — Mise à jour requise",
        f"Madame, Monsieur,\n\n"
        f"Votre attestation de vigilance URSSAF a expiré le {_past_date()}. "
        f"Sans renouvellement, vos clients ne pourront plus vérifier votre "
        f"conformité sociale.\n\n"
        f"Renouvelez votre attestation :\n"
        f"https://urssaf-attestation.com/{fake.lexify('????????')}\n\n"
        f"Vous devrez renseigner :\n"
        f"- Votre numéro SIRET ({_siret()})\n"
        f"- Votre RIB professionnel\n"
        f"- Une pièce d'identité\n\n"
        f"URSSAF — Service Attestations",
        "urssaf_cotisation", "URSSAF"
    ),

    # ── Ameli / Assurance Maladie ──────────────────────────────────────
    lambda: (
        f"Remboursement santé en attente — {_amount(23, 450)}",
        f"Cher(e) assuré(e),\n\n"
        f"Un remboursement de {_amount(23, 450)} est en attente sur votre "
        f"compte Ameli suite à vos derniers soins médicaux.\n\n"
        f"Pour finaliser ce virement, confirmez vos coordonnées bancaires :\n"
        f"https://ameli-remboursement-sante.fr/{fake.lexify('????????')}\n\n"
        f"Numéro de sécurité sociale : {_secu()}\n\n"
        f"Ce lien expire dans {_deadline()}.\n\n"
        f"L'Assurance Maladie",
        "ameli_sante", "Ameli"
    ),
    lambda: (
        f"Carte Vitale — Renouvellement obligatoire",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Votre Carte Vitale V3 doit être renouvelée avant le {_date_fr()} "
        f"conformément aux nouvelles dispositions légales.\n\n"
        f"Sans renouvellement, vos remboursements de soins seront suspendus.\n\n"
        f"Commandez votre nouvelle carte :\n"
        f"https://carte-vitale-renouvellement.fr/{fake.lexify('??????')}\n\n"
        f"Documents requis : pièce d'identité, RIB, photo récente.\n\n"
        f"CPAM de {_city()}",
        "ameli_sante", "Ameli"
    ),
    lambda: (
        f"Nouveau : Espace Mon Compte Ameli — Activation requise",
        f"Bonjour,\n\n"
        f"Suite à la migration de notre plateforme, votre espace Mon Compte "
        f"Ameli doit être réactivé. Sans activation avant le {_date_fr()}, "
        f"vous perdrez l'accès à vos relevés de remboursement.\n\n"
        f"→ Réactiver mon compte : https://mon-compte-ameli.org/{fake.lexify('????????')}\n\n"
        f"Munissez-vous de votre numéro de sécurité sociale et de votre RIB.\n\n"
        f"Service Numérique — Ameli.fr",
        "ameli_sante", "Ameli"
    ),

    # ── CAF ───────────────────────────────────────────────────────────
    lambda: (
        f"Suspension de vos droits CAF — Déclaration manquante",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Votre déclaration trimestrielle de ressources n'a pas été transmise. "
        f"Vos allocations (APL, RSA, Prime d'activité) seront suspendues "
        f"le {_date_fr()}.\n\n"
        f"Numéro d'allocataire : {fake.numerify('#######')}\n\n"
        f"Complétez votre déclaration immédiatement :\n"
        f"https://caf-declaration-urgente.fr/{fake.lexify('????????')}\n\n"
        f"Délai restant : {_deadline()}\n\n"
        f"CAF de {_city()}",
        "caf_allocation", "CAF"
    ),
    lambda: (
        f"Aide exceptionnelle de {_amount(150, 600)} — Confirmez votre éligibilité",
        f"Bonjour,\n\n"
        f"Dans le cadre du plan de soutien aux familles, la CAF vous "
        f"accorde une aide exceptionnelle de {_amount(150, 600)}.\n\n"
        f"Pour recevoir ce versement, vérifiez votre identité et vos "
        f"coordonnées bancaires sous {_deadline()} :\n\n"
        f"https://caf-aide-exceptionnelle.fr/{fake.lexify('??????')}\n\n"
        f"Cette aide est soumise à conditions de ressources.\n\n"
        f"CAF Nationale",
        "caf_allocation", "CAF"
    ),

    # ── La Poste / Chronopost ─────────────────────────────────────────
    lambda: (
        f"Colis en attente — Frais de réexpédition {_small_amount()}",
        f"Bonjour {fake.first_name()},\n\n"
        f"Votre colis n° {fake.numerify('## ### ### ####')} n'a pas pu être livré "
        f"le {_past_date()}. Motif : boîte aux lettres inaccessible.\n\n"
        f"Pour reprogrammer la livraison, réglez les frais de {_small_amount()} :\n"
        f"https://laposte-releve-colis.fr/{fake.lexify('????????')}\n\n"
        f"Sans action sous {_deadline()}, votre colis sera retourné.\n\n"
        f"La Poste — Colissimo",
        "laposte_colis", "La Poste"
    ),
    lambda: (
        f"Chronopost — Droits de douane à acquitter",
        f"Madame, Monsieur,\n\n"
        f"Votre colis en provenance de l'étranger (réf. {_ref()}) est retenu "
        f"par les services douaniers. Montant des droits : {_amount(8, 45)}.\n\n"
        f"Acquittez les droits pour libérer votre colis :\n"
        f"https://chronopost-douane-paiement.fr/{fake.lexify('??????')}\n\n"
        f"Délai de conservation : {_deadline()}.\n\n"
        f"Chronopost International",
        "laposte_colis", "Chronopost"
    ),
    lambda: (
        f"Colissimo — Nouvelle tentative de livraison",
        f"Cher(e) client(e),\n\n"
        f"Le facteur s'est présenté le {_past_date()} mais n'a pas pu "
        f"effectuer la livraison de votre colis ({fake.numerify('## ### ### ####')}).\n\n"
        f"Choisissez un créneau de livraison et confirmez votre adresse :\n"
        f"https://colissimo-livraison.fr/{fake.lexify('????????')}\n\n"
        f"Attention : une participation de {_small_amount()} est demandée "
        f"pour la seconde présentation.\n\n"
        f"La Poste — Service Livraison",
        "laposte_colis", "La Poste"
    ),

    # ── Banking ───────────────────────────────────────────────────────
    lambda: (
        f"Alerte sécurité — Activité suspecte sur votre compte {_bank()}",
        f"Cher(e) client(e),\n\n"
        f"Nous avons détecté une activité inhabituelle sur votre compte "
        f"bancaire (IP : {fake.ipv4()}, localisation : "
        f"{random.choice(['Lagos', 'Moscou', 'Bucarest', 'Istanbul', 'Shenzhen'])}).\n\n"
        f"Si vous n'êtes pas à l'origine de cette activité, sécurisez "
        f"immédiatement votre compte :\n"
        f"https://securite-{fake.lexify('???')}-banque.fr/{fake.lexify('????????')}\n\n"
        f"Sans vérification sous {_deadline()}, votre compte sera bloqué.\n\n"
        f"Service Fraude — {_bank()}",
        "banque_securite", "Banque"
    ),
    lambda: (
        f"Virement de {_amount(800, 5000)} — Confirmation requise",
        f"Bonjour,\n\n"
        f"Un virement de {_amount(800, 5000)} vers le compte "
        f"{_iban_fr()} a été initié le {_past_date()} depuis votre espace.\n\n"
        f"Si vous n'avez pas autorisé cette opération, bloquez-la "
        f"immédiatement :\n"
        f"https://opposition-virement.fr/{fake.lexify('??????')}\n\n"
        f"Vous disposez de {_deadline()} pour faire opposition.\n\n"
        f"Service Opposition — {_bank()}",
        "banque_securite", "Banque"
    ),
    lambda: (
        f"Nouvelle réglementation DSP2 — Mise à jour obligatoire",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Suite à l'entrée en vigueur de la directive européenne DSP2, "
        f"votre dispositif d'authentification forte doit être mis à jour "
        f"avant le {_date_fr()}.\n\n"
        f"Sans cette mise à jour :\n"
        f"- Vos paiements par carte seront refusés\n"
        f"- L'accès à votre espace en ligne sera restreint\n\n"
        f"→ Mettre à jour : https://dsp2-auth-update.fr/{fake.lexify('????????')}\n\n"
        f"{_bank()} — Service Monétique",
        "banque_securite", "Banque"
    ),

    # ── FranceConnect / Identity ──────────────────────────────────────
    lambda: (
        f"FranceConnect — Connexion depuis un appareil inconnu",
        f"Bonjour,\n\n"
        f"Une connexion à votre compte FranceConnect a été effectuée le "
        f"{_past_date()} à {random.randint(1,23)}h{random.randint(10,59)} depuis "
        f"{random.choice(['un appareil Android à Abidjan', 'un PC Windows à Saint-Pétersbourg', 'un iPhone à Casablanca', 'un Mac à Beijing'])}.\n\n"
        f"Si ce n'était pas vous, changez votre mot de passe immédiatement :\n"
        f"https://franceconnect-securite-compte.fr/{fake.lexify('????????')}\n\n"
        f"Votre identité numérique est en danger.\n\n"
        f"FranceConnect — Sécurité",
        "franceconnect_id", "FranceConnect"
    ),
    lambda: (
        f"Service-Public.fr — Vérification d'identité urgente",
        f"Madame, Monsieur,\n\n"
        f"Suite à un signalement de fraude, nous devons vérifier votre identité "
        f"pour maintenir l'accès à vos démarches administratives en ligne.\n\n"
        f"Transmettez les documents suivants avant le {_date_fr()} :\n"
        f"1. Carte nationale d'identité (recto/verso)\n"
        f"2. Justificatif de domicile de moins de 3 mois\n"
        f"3. Dernier avis d'imposition\n\n"
        f"→ https://service-public-verification.fr/{fake.lexify('??????')}\n\n"
        f"DILA — Direction de l'Information Légale",
        "franceconnect_id", "Service-Public"
    ),

    # ── Invoice / Telecom ─────────────────────────────────────────────
    lambda: (
        f"Facture {_telecom()} impayée — Dernière relance",
        f"Madame, Monsieur,\n\n"
        f"Votre facture du {_past_date()} d'un montant de {_amount(29, 89)} "
        f"est toujours impayée malgré nos précédentes relances.\n\n"
        f"Sans règlement dans les {_deadline()}, nous procéderons à :\n"
        f"- La suspension de votre ligne\n"
        f"- La transmission de votre dossier à un huissier de justice\n"
        f"- L'inscription au fichier des incidents de paiement\n\n"
        f"Réglez votre facture :\n"
        f"https://paiement-facture-{fake.lexify('???')}.fr/{fake.lexify('??????')}\n\n"
        f"Réf. client : {_ref()}\n\n"
        f"Service Contentieux — {_telecom()}",
        "facture_paiement", "Télécom"
    ),
    lambda: (
        f"EDF — Coupure programmée sans régularisation",
        f"Cher(e) client(e),\n\n"
        f"Nous constatons un impayé de {_amount(85, 350)} sur votre contrat "
        f"d'électricité (n° {fake.numerify('#### #### ####')}).\n\n"
        f"Une coupure d'alimentation est programmée le {_date_fr()} si "
        f"aucun règlement n'est effectué.\n\n"
        f"→ Régulariser : https://edf-paiement.com/{fake.lexify('????????')}\n\n"
        f"EDF — Service Recouvrement",
        "facture_paiement", "EDF"
    ),

    # ── BEC (Business Email Compromise) — Auto-entrepreneurs ──────────
    lambda: (
        f"Commande urgente — Devis requis sous {_deadline()}",
        f"Bonjour {fake.first_name()},\n\n"
        f"Je suis {fake.name()}, directeur(rice) des achats chez "
        f"{fake.company()}. Nous avons un besoin urgent pour un projet "
        f"confidentiel.\n\n"
        f"Pourriez-vous me transmettre un devis pour la prestation suivante :\n"
        f"- {random.choice(['Développement site web', 'Conseil en communication', 'Formation interne', 'Audit comptable', 'Design graphique'])}\n"
        f"- Budget estimé : {_amount(2000, 15000)}\n"
        f"- Délai : {random.randint(5, 15)} jours\n\n"
        f"Merci de répondre rapidement, c'est très urgent et confidentiel.\n\n"
        f"Cordialement,\n{fake.name()}\n"
        f"{fake.job()}\n{fake.company()}",
        "bec_autoentrepreneur", "BEC"
    ),
    lambda: (
        f"Changement de RIB — Client {fake.company()}",
        f"Bonjour,\n\n"
        f"Je me permets de vous contacter car nous avons récemment changé "
        f"de banque. Pourriez-vous mettre à jour notre RIB pour les "
        f"prochains virements ?\n\n"
        f"Nouveau RIB :\n"
        f"IBAN : {_iban_fr()}\n"
        f"BIC : {fake.lexify('????????').upper()}\n"
        f"Titulaire : {fake.company()}\n\n"
        f"Merci de confirmer la prise en compte par retour de mail.\n\n"
        f"Cordialement,\n{fake.name()}\n"
        f"Comptabilité — {fake.company()}",
        "bec_autoentrepreneur", "BEC"
    ),
]

print(f"Phishing templates defined: {len(PHISHING_TEMPLATES)}")

Phishing templates defined: 23


## 3. Legitimate Email Templates

We also generate **legitimate French emails** to balance the dataset.
These represent real emails a French auto-entrepreneur would receive.

In [4]:
# ── LEGITIMATE EMAIL TEMPLATES ────────────────────────────────────────

LEGIT_TEMPLATES: list = [
    # ── Administrative / Government ────────────────────────────────────
    lambda: (
        f"Votre attestation URSSAF est disponible",
        f"Bonjour,\n\n"
        f"Votre attestation de vigilance est disponible dans votre espace "
        f"personnel sur urssaf.fr. Elle est valable 6 mois à compter du "
        f"{_date_fr()}.\n\n"
        f"Connectez-vous à votre espace autoentrepreneur.urssaf.fr pour "
        f"la télécharger.\n\n"
        f"Cordialement,\nURSSAF",
        "admin_legit", "URSSAF"
    ),
    lambda: (
        f"Confirmation de déclaration trimestrielle",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Nous accusons réception de votre déclaration trimestrielle de "
        f"chiffre d'affaires pour la période du {_past_date()} au {_date_fr()}.\n\n"
        f"Montant déclaré : {_amount(500, 8000)}\n"
        f"Cotisations dues : {_amount(50, 800)}\n"
        f"Prélèvement prévu le : {_date_fr()}\n\n"
        f"Si ces informations sont erronées, contactez-nous au 3698.\n\n"
        f"URSSAF — Service Auto-Entrepreneurs",
        "admin_legit", "URSSAF"
    ),
    lambda: (
        f"Avis d'imposition 2025 disponible",
        f"Cher(e) contribuable,\n\n"
        f"Votre avis d'imposition sur les revenus de 2024 est disponible "
        f"dans votre espace particulier sur impots.gouv.fr.\n\n"
        f"Pour y accéder, connectez-vous avec votre numéro fiscal "
        f"et votre mot de passe.\n\n"
        f"Direction Générale des Finances Publiques",
        "admin_legit", "DGFiP"
    ),

    # ── Banking ───────────────────────────────────────────────────────
    lambda: (
        f"Récapitulatif mensuel — {_bank()}",
        f"Bonjour {fake.first_name()},\n\n"
        f"Votre relevé de compte du mois est disponible dans votre espace "
        f"client {_bank()}.\n\n"
        f"Solde au {_past_date()} : {_amount(500, 12000)}\n"
        f"Prochain prélèvement : {_date_fr()}\n\n"
        f"Votre conseiller : {fake.name()}\nAgence de {_city()}",
        "banking_legit", "Banque"
    ),
    lambda: (
        f"Confirmation de virement effectué",
        f"Madame, Monsieur,\n\n"
        f"Votre virement de {_amount(100, 5000)} vers le compte de "
        f"{fake.name()} a été effectué avec succès le {_past_date()}.\n\n"
        f"Référence : {_ref()}\n\n"
        f"Si vous n'êtes pas à l'origine de ce virement, contactez "
        f"votre agence au {fake.phone_number()}.\n\n"
        f"{_bank()}",
        "banking_legit", "Banque"
    ),

    # ── E-commerce / Delivery ──────────────────────────────────────────
    lambda: (
        f"Votre commande a été expédiée — {_ref()}",
        f"Bonjour {fake.first_name()},\n\n"
        f"Nous avons le plaisir de vous informer que votre commande "
        f"({_ref()}) a été expédiée le {_past_date()}.\n\n"
        f"Suivi Colissimo : {fake.numerify('## ### ### ####')}\n"
        f"Livraison estimée : {_date_fr()}\n\n"
        f"L'équipe {fake.company()}",
        "ecommerce_legit", "E-commerce"
    ),
    lambda: (
        f"Facture n° {_ref()} — {fake.company()}",
        f"Madame, Monsieur {fake.last_name()},\n\n"
        f"Veuillez trouver ci-joint la facture n° {_ref()} pour un montant "
        f"TTC de {_amount()}.\n\n"
        f"Date d'émission : {_past_date()}\n"
        f"Date d'échéance : {_date_fr()}\n"
        f"Mode de paiement : virement bancaire\n\n"
        f"Cordialement,\n{fake.name()}\n{fake.company()}",
        "business_legit", "Entreprise"
    ),

    # ── Professional / Networking ──────────────────────────────────────
    lambda: (
        f"Confirmation de rendez-vous — {_date_fr()}",
        f"Bonjour {fake.first_name()},\n\n"
        f"Je vous confirme notre rendez-vous le {_date_fr()} à "
        f"{random.randint(8,18)}h{random.choice(['00','15','30','45'])} "
        f"à {random.choice(['nos bureaux', 'votre cabinet', 'en visioconférence'])} "
        f"pour discuter de {random.choice(['votre projet', 'notre collaboration', 'le contrat', 'la proposition commerciale'])}.\n\n"
        f"N'hésitez pas à me contacter si vous avez des questions.\n\n"
        f"Cordialement,\n{fake.name()}\n{fake.job()}\n{fake.company()}",
        "professional_legit", "Professionnel"
    ),
    lambda: (
        f"Newsletter Simplon.co — Formations {random.choice(['Mars','Avril','Mai','Juin'])} 2026",
        f"Bonjour,\n\n"
        f"Découvrez nos prochaines formations en {_city()} :\n\n"
        f"• Développeur Web — Début : {_date_fr()}\n"
        f"• Data Analyst — Début : {_date_fr()}\n"
        f"• DevOps / Cloud — Début : {_date_fr()}\n\n"
        f"Toutes nos formations sont gratuites et ouvertes à tous.\n\n"
        f"En savoir plus sur simplon.co\n\n"
        f"L'équipe Simplon",
        "newsletter_legit", "Newsletter"
    ),
    lambda: (
        f"Doctolib — Rappel de rendez-vous",
        f"Bonjour {fake.first_name()},\n\n"
        f"Vous avez rendez-vous avec Dr {fake.last_name()} "
        f"({random.choice(['Médecin généraliste', 'Dentiste', 'Ophtalmologue', 'Kinésithérapeute'])}) "
        f"le {_date_fr()} à {random.randint(8,19)}h{random.choice(['00','15','30','45'])}.\n\n"
        f"Adresse : {fake.address()}\n\n"
        f"Pensez à apporter votre carte Vitale.\n\n"
        f"L'équipe Doctolib",
        "health_legit", "Doctolib"
    ),

    # ── Telecom ───────────────────────────────────────────────────────
    lambda: (
        f"Votre facture {_telecom()} est disponible",
        f"Bonjour,\n\n"
        f"Votre facture du mois de {random.choice(['janvier','février','mars','avril','mai'])} "
        f"est disponible dans votre espace client.\n\n"
        f"Montant : {_amount(19, 65)}\n"
        f"Prélèvement le : {_date_fr()}\n\n"
        f"Consultez votre facture en détail sur l'application {_telecom()}.\n\n"
        f"{_telecom()} — Service Client",
        "telecom_legit", "Télécom"
    ),
]

print(f"Legitimate templates defined: {len(LEGIT_TEMPLATES)}")

Legitimate templates defined: 11


## 4. Generate Synthetic Dataset

In [5]:
# ── Generate phishing emails ──────────────────────────────────────────

phishing_records: list[dict] = []

for i in range(PHISHING_TARGET):
    template_fn = random.choice(PHISHING_TEMPLATES)
    subject, body, archetype, fr_entity = template_fn()
    
    full_text: str = f"Objet : {subject}\n\n{body}"
    
    phishing_records.append({
        "text": full_text,
        "label": 1,
        "source": "synthetic_fr",
        "language": "fr",
        "archetype": archetype,
        "fr_entity": fr_entity,
    })

print(f"Phishing emails generated: {len(phishing_records):,}")

# ── Generate legitimate emails ─────────────────────────────────────────

legit_records: list[dict] = []

for i in range(LEGIT_TARGET):
    template_fn = random.choice(LEGIT_TEMPLATES)
    subject, body, archetype, fr_entity = template_fn()
    
    full_text: str = f"Objet : {subject}\n\n{body}"
    
    legit_records.append({
        "text": full_text,
        "label": 0,
        "source": "synthetic_fr",
        "language": "fr",
        "archetype": archetype,
        "fr_entity": fr_entity,
    })

print(f"Legitimate emails generated: {len(legit_records):,}")

# ── Combine ──────────────────────────────────────────────────────────
all_synthetic: list[dict] = phishing_records + legit_records
df_synthetic: pd.DataFrame = pd.DataFrame(all_synthetic)

print(f"\nTotal synthetic: {len(df_synthetic):,}")
print(f"\nLabel distribution:")
print(df_synthetic["label"].value_counts())
print(f"\nArchetype distribution (phishing):")
print(df_synthetic[df_synthetic["label"] == 1]["archetype"].value_counts())
print(f"\nArchetype distribution (legit):")
print(df_synthetic[df_synthetic["label"] == 0]["archetype"].value_counts())

Phishing emails generated: 2,000
Legitimate emails generated: 1,000

Total synthetic: 3,000

Label distribution:
label
1    2000
0    1000
Name: count, dtype: int64

Archetype distribution (phishing):
archetype
dgfip_tax               273
banque_securite         268
urssaf_cotisation       268
laposte_colis           264
ameli_sante             250
franceconnect_id        185
caf_allocation          169
facture_paiement        162
bec_autoentrepreneur    161
Name: count, dtype: int64

Archetype distribution (legit):
archetype
admin_legit           282
banking_legit         200
newsletter_legit       97
business_legit         90
telecom_legit          90
ecommerce_legit        88
health_legit           77
professional_legit     76
Name: count, dtype: int64


In [6]:
# ── Preview samples ───────────────────────────────────────────────────
print("=" * 70)
print("PHISHING SAMPLE:")
print("=" * 70)
sample_p = df_synthetic[df_synthetic["label"] == 1].sample(1, random_state=42).iloc[0]
print(f"Archetype: {sample_p['archetype']} | Entity: {sample_p['fr_entity']}")
print("-" * 70)
print(sample_p["text"][:500])

print("\n" + "=" * 70)
print("LEGITIMATE SAMPLE:")
print("=" * 70)
sample_l = df_synthetic[df_synthetic["label"] == 0].sample(1, random_state=42).iloc[0]
print(f"Archetype: {sample_l['archetype']} | Entity: {sample_l['fr_entity']}")
print("-" * 70)
print(sample_l["text"][:500])

PHISHING SAMPLE:
Archetype: urssaf_cotisation | Entity: URSSAF
----------------------------------------------------------------------
Objet : Cotisations impayées — Mise en demeure CMD-7733-7304-76

Madame, Monsieur Moreau,

Nous constatons que vos cotisations sociales du trimestre en cours (montant : 1 891,59 €) n'ont pas été réglées à la date d'échéance.

Numéro de cotisant : 805 502 945 693
SIRET : 137 661 219 52145

Sans régularisation dans les 72 heures, votre dossier sera transmis au service de recouvrement forcé et des majorations de 5 % par mois seront appliquées.

Régularisez en ligne :
https://urssaf-cotisations.fr/F

LEGITIMATE SAMPLE:
Archetype: newsletter_legit | Entity: Newsletter
----------------------------------------------------------------------
Objet : Newsletter Simplon.co — Formations Juin 2026

Bonjour,

Découvrez nos prochaines formations en Saint Marcel :

• Développeur Web — Début : 25/02/2026
• Data Analyst — Début : 22/02/2026
• DevOps / Cloud — Début : 10/0

## 5. Deduplication & Quality Checks

In [7]:
# ── Dedup by SHA-256 ──────────────────────────────────────────────────
df_synthetic["text_hash"] = df_synthetic["text"].apply(
    lambda t: hashlib.sha256(t.encode()).hexdigest()
)

before: int = len(df_synthetic)
df_synthetic = df_synthetic.drop_duplicates(subset=["text_hash"]).reset_index(drop=True)
after: int = len(df_synthetic)

print(f"Before dedup : {before:,}")
print(f"After dedup  : {after:,}")
print(f"Removed      : {before - after:,} duplicates")

Before dedup : 3,000
After dedup  : 2,863
Removed      : 137 duplicates


In [8]:
# ── Quality metrics ───────────────────────────────────────────────────

df_synthetic["text_len"] = df_synthetic["text"].str.len()
print("Text length statistics:")
print(df_synthetic["text_len"].describe())

# French markers
french_markers: list[str] = [
    "vous", "votre", "veuillez", "cordialement",
    "madame", "monsieur", "bonjour",
    "é", "è", "ê", "à", "ç", "ô",
]

def french_score(text: str) -> int:
    t: str = text.lower()
    return sum(1 for m in french_markers if m in t)

df_synthetic["fr_score"] = df_synthetic["text"].apply(french_score)

print(f"\nFrench marker presence:")
print(f"  Min markers    : {df_synthetic['fr_score'].min()}")
print(f"  Mean markers   : {df_synthetic['fr_score'].mean():.1f}")
print(f"  All ≥2 markers : {(df_synthetic['fr_score'] >= 2).all()}")

# Phishing-specific: urgency check
urgency_words: list[str] = [
    "urgent", "immédiat", "délai", "suspension", "bloqué",
    "pénalité", "dernière", "obligatoire", "sans action", "recouvrement",
]
phishing_mask = df_synthetic["label"] == 1
urgency_rate: float = df_synthetic.loc[phishing_mask, "text"].apply(
    lambda t: any(w in t.lower() for w in urgency_words)
).mean()

print(f"\n  Phishing with urgency markers     : {urgency_rate:.1%}")
print(f"  Unique archetypes (phishing)       : {df_synthetic.loc[phishing_mask, 'archetype'].nunique()}")
print(f"  Unique archetypes (legit)          : {df_synthetic.loc[~phishing_mask, 'archetype'].nunique()}")

Text length statistics:
count    2863.000000
mean      386.539644
std        74.237821
min       230.000000
25%       335.000000
50%       392.000000
75%       440.000000
max       562.000000
Name: text_len, dtype: float64

French marker presence:
  Min markers    : 2
  Mean markers   : 5.0
  All ≥2 markers : True

  Phishing with urgency markers     : 59.3%
  Unique archetypes (phishing)       : 9
  Unique archetypes (legit)          : 8


## 6. Export

In [9]:
# ── Export to CSV ──────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
filename: str = f"synthetic_fr_emails_{len(df_synthetic)}_{timestamp}.csv"
output_path: Path = OUTPUT_DIR / filename

export_cols: list[str] = [
    "text", "label", "source", "language", "archetype",
    "fr_entity", "text_hash",
]

df_synthetic[export_cols].to_csv(output_path, index=False, encoding="utf-8")

# Stable-name copy for notebook 09
stable_path: Path = OUTPUT_DIR / "synthetic_fr_emails.csv"
df_synthetic[export_cols].to_csv(stable_path, index=False, encoding="utf-8")

size_kb: float = output_path.stat().st_size / 1024
print(f"Exported       : {output_path}")
print(f"Stable copy    : {stable_path}")
print(f"Rows           : {len(df_synthetic):,}")
print(f"  Phishing     : {(df_synthetic['label'] == 1).sum():,}")
print(f"  Legitimate   : {(df_synthetic['label'] == 0).sum():,}")
print(f"Size           : {size_kb:.1f} KB")
print(f"Columns        : {export_cols}")

Exported       : data/raw/db/synthetic_fr_emails_2863_20260228.csv
Stable copy    : data/raw/db/synthetic_fr_emails.csv
Rows           : 2,863
  Phishing     : 2,000
  Legitimate   : 863
Size           : 1423.3 KB
Columns        : ['text', 'label', 'source', 'language', 'archetype', 'fr_entity', 'text_hash']


## 7. Summary

### What this notebook demonstrates

| Criterion | Evidence |
|-----------|----------|
| **Synthetic data generation** | Template-based + Faker(fr_FR) for maximum variability |
| **French specificity** | 8 phishing archetypes + BEC targeting auto-entrepreneurs |
| **Class balance** | 2:1 phishing-to-legit ratio (configurable) |
| **Entity diversity** | DGFiP, URSSAF, Ameli, CAF, La Poste, banks, telecom, FranceConnect |
| **Legitimate coverage** | Admin, banking, e-commerce, professional, healthcare, telecom |
| **RGPD compliance** | 100% synthetic — no real PII |
| **Quality** | Dedup + French marker validation + urgency detection |

### Defence talking point

> *"Since no public French phishing corpus exists, we generate synthetic data*
> *that mirrors the real French threat landscape: URSSAF cotisation fraud,*
> *DGFiP tax scams, Ameli/CAF benefit bait, La Poste delivery phishing,*
> *and BEC targeting auto-entrepreneurs. Each email uses formal 'vous' register,*
> *realistic reference numbers (SIRET, numéro de sécu, IBAN), and French*
> *administrative vocabulary. The templates produce 3 000+ unique emails*
> *with Faker variability — no two emails are identical."*

### Next step

→ **Notebook 09** loads both adapted (notebook 10) + synthetic (this notebook) data  
&nbsp;&nbsp;into SQLite for the C1 *"base de données"* extraction requirement.